## Setup

In [ ]:
import optuna
from optuna.samplers import TPESampler
from optuna.trial import Trial
from optuna.study import create_study

from libs import constants as Cs
import numpy as np
from concurrent.futures import ProcessPoolExecutor
from scipy.spatial.distance import pdist


SEEDS = [101, 102, 103]
TEST_EVAL_EPS = 5

## Add Novelty Diff

In [ ]:
def objective(trial:Trial):
    env = Cs.ENIVROMENTS["lunarlander"]()
    l = trial.suggest_categorical("lambda",[ 40, 50, 60, 70])
    mr = trial.suggest_float("mutation_rate", 0, 0.5, step=0.01)
    cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
    fit_w = trial.suggest_float("start_fit_w", 0.2, 1, step=0.1)
    decay = trial.suggest_float("decay", 0.5, 5, step=0.5)

    with ProcessPoolExecutor(max_workers=len(SEEDS)) as executor:
        futures = [
            executor.submit(Cs.diff_cont.argumented_function, 
                env="lunarlander",
                container="add_novelty",
                ng = 15,
                l = l,
                cr = cr,
                mr = mr,
                fitness_weight=fit_w,
                decay=decay,
                seed = seed) for seed in SEEDS
        ]

        pops = [f.result()[1] for f in futures]
        fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for pop in pops for p in pop ]
    #trial.set_user_attr("scores", fitnesses)
    fitnesses = list(map(lambda x:x[0], fitnesses))
    return np.max(fitnesses)


sampler = TPESampler(n_startup_trials=15)
study = create_study(direction="maximize", sampler=sampler, study_name="diff_add_novelty_exp_fixed_last", storage="sqlite:///Data/optuna/lunarlander/add_novelty/diff_fixed.db", load_if_exists=True)
study.optimize(objective, n_trials=160, n_jobs=5)
print(study.best_trials)

## Sub Novelty

In [ ]:
def objective(trial:Trial):
    env = Cs.ENIVROMENTS["lunarlander"]()
    l = trial.suggest_categorical("lambda",[ 40, 50, 60, 70])
    mr = trial.suggest_float("mutation_rate", 0, 0.5, step=0.01)
    cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
    fit_w = trial.suggest_float("start_fit_w", 0.2, 1, step=0.1)
    decay = trial.suggest_float("decay", 0.5, 5, step=0.5)

    with ProcessPoolExecutor(max_workers=len(SEEDS)) as executor:
        futures = [
            executor.submit(Cs.diff_cont.argumented_function, 
                env="lunarlander",
                container="sub_novelty",
                ng = 15,
                l = l,
                cr = cr,
                mr = mr,
                fitness_weight=fit_w,
                decay=decay,
                seed = seed) for seed in SEEDS
        ]

        pops = [f.result()[1] for f in futures]
        fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for pop in pops for p in pop ]
    #trial.set_user_attr("scores", fitnesses)
    behaviors = list(map(lambda x:x[1], fitnesses))
    fitnesses = list(map(lambda x:x[0], fitnesses))
    diversity = pdist(np.array(behaviors)).mean()
    assert diversity > 0
    return np.max(fitnesses) + diversity


sampler = TPESampler(n_startup_trials=15)
study = create_study(direction="maximize", sampler=sampler, study_name="diff_sub_novelty_exp_repair_last", storage="sqlite:///Data/optuna/lunarlander/sub_novelty/diff.db", load_if_exists=True)
study.optimize(objective, n_trials=170, n_jobs=5)
print(study.best_trials)

## Fit_Archiving

In [ ]:
def objective(trial:Trial):
    environment = "lunarlander"
    env = Cs.ENIVROMENTS[environment]()
    l = trial.suggest_categorical("lambda",[ 40, 50, 60, 70])
    mr = trial.suggest_float("mutation_rate", 0, 1, step=0.1)
    cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
    archiving = trial.suggest_int("archiving_period", 2, 5)
    archive_batch = trial.suggest_int("archive_batch", 1, 5)
    with ProcessPoolExecutor(max_workers=len(SEEDS)) as executor:
        futures = [
            executor.submit(Cs.diff_cont.argumented_function, env=environment,
                container="fit_archiving",
                ng = 15,
                l = l,
                cr = cr,
                mr = mr,
                archiving_period=archiving,
                archive_batch=archive_batch,
                seed = seed) for seed in SEEDS
        ]

        pops = [f.result()[1] for f in futures]
        fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for pop in pops for p in pop ]


    #trial.set_user_attr("scores", fitnesses)
    fitnesses = list(map(lambda x:x[0], fitnesses))
    return np.max(fitnesses)


sampler = TPESampler(n_startup_trials=20)
diff_fita_study = create_study(direction="maximize", study_name="diff_fit_archiving_reloaded", sampler=sampler, storage="sqlite:///Data/optuna/lunarlander/fit_archiving/diff.db", load_if_exists=True)
diff_fita_study.optimize(objective, n_trials=150, n_jobs=5)
print(diff_fita_study.best_trials)

## Novlety Limit Archiving

In [ ]:
def objective(trial:Trial):
    environment = "lunarlander"
    env = Cs.ENIVROMENTS[environment]()
    l = trial.suggest_categorical("lambda",[ 40, 50, 60, 70])
    mr = trial.suggest_float("mutation_rate", 0, 1, step=0.1)
    cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
    archiving = trial.suggest_int("archiving_period", 2, 5)
    archive_batch = trial.suggest_int("archive_batch", 1, 5)
    limit = trial.suggest_float("limit", -200, -50, step=10)

    with ProcessPoolExecutor(max_workers=len(SEEDS)) as executor:
        futures = [
            executor.submit(Cs.diff_cont.argumented_function, env=environment,
                container="novelty_limit",
                ng = 15,
                l = l,
                cr = cr,
                mr = mr,
                archiving_period=archiving,
                archive_batch=archive_batch,
                limit=limit,
                seed = seed) for seed in SEEDS
        ]

        pops = [f.result()[1] for f in futures]
        fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for pop in pops for p in pop ]


    #trial.set_user_attr("scores", fitnesses)
    fitnesses = list(map(lambda x:x[0], fitnesses))
    return np.max(fitnesses)


sampler = TPESampler(n_startup_trials=20)
diff_fita_study = create_study(direction="maximize", study_name="diff_novelty_limit", sampler=sampler, storage="sqlite:///Data/optuna/lunarlander/novelty_limit/diff.db", load_if_exists=True)
diff_fita_study.optimize(objective, n_trials=170, n_jobs=5)
print(diff_fita_study.best_trials)

## Novelty Archiving

In [ ]:
def objective(trial:Trial):
    environment = "lunarlander"
    env = Cs.ENIVROMENTS[environment]()
    l = trial.suggest_categorical("lambda",[ 40, 50, 60, 70])
    mr = trial.suggest_float("mutation_rate", 0, 1, step=0.1)
    cr = trial.suggest_float("cross_rate", 0.3, 1, step=0.1)
    archiving = trial.suggest_int("archiving_period", 2, 5)
    archive_batch = trial.suggest_int("archive_batch", 1, 5)
    with ProcessPoolExecutor(max_workers=len(SEEDS)) as executor:
        futures = [
            executor.submit(Cs.diff_cont.argumented_function, env=environment,
                container="novelty_archiving",
                ng = 15,
                l = l,
                cr = cr,
                mr = mr,
                archiving_period=archiving,
                archive_batch=archive_batch,
                seed = seed) for seed in SEEDS
        ]

        pops = [f.result()[1] for f in futures]
        fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for pop in pops for p in pop ]


    #trial.set_user_attr("scores", fitnesses)
    fitnesses = list(map(lambda x:x[0], fitnesses))
    return np.max(fitnesses)


sampler = TPESampler(n_startup_trials=20)
diff_fita_study = create_study(direction="maximize", study_name="diff_novelty_archiving", sampler=sampler, storage="sqlite:///Data/optuna/lunarlander/novelty_archiving/diff.db", load_if_exists=True)
diff_fita_study.optimize(objective, n_trials=150, n_jobs=5)
print(diff_fita_study.best_trials)